# 📦 Cubagem Multimodal v5

**Correções e melhorias em relação à v4:**

| Problema | Correção |
|---|---|
| CPU em vez de GPU | Verificação explícita no início |
| OneCycleLR incompatível com early stopping | `ReduceLROnPlateau` (step por época) |
| HybridLoss calcula MAPE no espaço log (errado) | MAPE calculado após `expm1` (espaço real) |
| CrossModalAttention inútil com seq_len=1 | Removida |
| 883k params para 32k amostras → overfitting | Backbone menor, projeções mais enxutas |
| `main_category` pouco granular (44 classes) | `source_category` (mais discriminativa) |
| Embeddings congelados = teto de performance | **Fase 2**: últimas 4 camadas do RoBERTa descongeladas |
| Mean pooling ignora tokens de dimensão | **Fase 2**: attention pooling aprendida |

**Estrutura:**
- **Fase 1** — MLP com embeddings pré-computados (rápido, baseline corrigido)
- **Fase 2** — Fine-tuning do RoBERTa com attention pooling (maior potencial)


## 0. Drive e verificação de GPU

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 1. Imports e configuração

In [4]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


In [5]:
df = pd.read_csv("../data/cubagem_40k_amazon.csv")
df.columns

Index(['asin', 'title', 'main_category', 'source_category', 'categories',
       'store', 'length_cm', 'width_cm', 'height_cm', 'weight_g', 'volume_cm3',
       'density_g_cm3', 'log_volume', 'log_weight', 'dim_max_cm', 'dim_min_cm',
       'dim_ratio', 'price_raw', 'price_numeric', 'log_price', 'avg_rating',
       'n_ratings', 'n_images', 'title_length', 'title_word_count',
       'has_dim_in_title', 'has_size_word_in_title', 'has_qty_in_title',
       'description', 'description_length', 'n_categories_levels', 'image_url',
       'split'],
      dtype='str')

In [6]:
features = {
    "categorical": [
        "main_category",
        "source_category",
        # Leaving those out because I dont want to deal with high cardinality now
        # "categories",
        # "store",
    ],
    "text": [
        "title",
        "description",
    ],
    "numerical": [
        "price_numeric",
        "avg_rating",
        "n_ratings",
        "n_images",
        "title_length",
        "title_word_count",
        "description_length",
        "n_categories_levels",
    ],
    "title_regex": [
        "has_dim_in_title",
        "has_size_word_in_title",
        "has_qty_in_title",
    ],
    "image": [
        "image_url",
    ],
}

targets = [
    "length_cm",   # length >= width >= height
    "width_cm",
    "height_cm",
    "weight_g",
]

log_targets = [
    "log_length_cm",   # length >= width >= height
    "log_width_cm",
    "log_height_cm",
    "log_weight_g",
]

In [7]:
df[log_targets] = np.log(df[targets])

In [8]:
ROOT = "../embeddings"

EMBEDDINGS = {
    "texto_train"  : f"{ROOT}/texto_train.npy",
    "texto_val"    : f"{ROOT}/texto_val.npy",
    "imagem_train" : f"{ROOT}/imagem_train.npy",
    "imagem_val"   : f"{ROOT}/imagem_val.npy",
}

## Dados e feature engineering

In [9]:
df = df.dropna(subset=targets + ["title", "image_url"])
df[features["title_regex"]] = df[features["title_regex"]].fillna(False).astype(float)

In [10]:
from sklearn.preprocessing import OneHotEncoder

def one_hot_encode(df, cols, drop_first=False):
    encoder = OneHotEncoder(
        sparse_output=False,
        drop="first" if drop_first else None,
        handle_unknown="ignore",  # unseen category at inference -> all zeros, no crash
    )
 
    encoded = encoder.fit_transform(df[cols])
    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(cols),
        index=df.index,
    )
 
    df_out = df.drop(columns=cols).join(encoded_df)
    return df_out, encoder
 
 
def apply_one_hot_encode(df, cols, encoder):
    encoded = encoder.transform(df[cols])
    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(cols),
        index=df.index,
    )
    return df.drop(columns=cols).join(encoded_df)
 


In [11]:
df_encoded, encoder = one_hot_encode(df, features["categorical"])
OHE_columns = [x for x in df_encoded.columns if x not in df.columns]

In [12]:
df_train = df_encoded[df_encoded["split"] == "train"].reset_index(drop=True)
df_val   = df_encoded[df_encoded["split"] == "val"  ].reset_index(drop=True)

print(f"Treino: {len(df_train):,} | Val: {len(df_val):,}")

Treino: 31,986 | Val: 7,999


In [31]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(max_iter=10, random_state=42)
# Fit and transform on train
df_train[features["numerical"]] = imputer.fit_transform(df_train[features["numerical"]])
# Just transform on val
df_val[features["numerical"]] = imputer.transform(df_val[features["numerical"]])

## Carrega Embeddings

In [32]:
class CubagemDataset(Dataset):
    def __init__(self, df, text_npz_path, img_npz_path):
        self.df = df.reset_index(drop=True)
        self.asins = self.df["asin"].values
        
        # Load embeddings
        text_data = np.load(text_npz_path, allow_pickle=True)
        img_data  = np.load(img_npz_path, allow_pickle=True)
        
        self.text_dict = dict(zip(text_data["asins"], text_data["embeddings"]))
        self.img_dict  = dict(zip(img_data["asins"], img_data["embeddings"]))
        
        self.num_feats   = torch.tensor(self.df[features["numerical"]].values, dtype=torch.float32)
        self.cat_feats   = torch.tensor(self.df[OHE_columns].values, dtype=torch.float32)
        self.regex_feats = torch.tensor(self.df[features["title_regex"]].values, dtype=torch.float32)
        self.targets     = torch.tensor(self.df[log_targets].values, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        asin = self.asins[idx]
        
        # Fetch embeddings dynamically
        emb_t = self.text_dict.get(asin, np.zeros(768, dtype=np.float32))
        emb_i = self.img_dict.get(asin, np.zeros(768, dtype=np.float32))
        
        return {
            "emb_texto":   torch.tensor(emb_t, dtype=torch.float32),
            "emb_img":     torch.tensor(emb_i, dtype=torch.float32),
            "num_feats":   self.num_feats[idx],
            "cat_feats":   self.cat_feats[idx],
            "regex_feats": self.regex_feats[idx],
            "targets":     self.targets[idx],
        }

Create dataloaders

In [33]:
BATCH_SIZE = 32

def create_loaders(df_tr, df_vl):
    
    ds_tr = CubagemDataset(df_tr, text_npz_path="../embeddings/text_embeddings.npz", img_npz_path="../embeddings/imagem_embeddings.npz")
    ds_vl = CubagemDataset(df_vl, text_npz_path="../embeddings/text_embeddings.npz", img_npz_path="../embeddings/imagem_embeddings.npz")
    
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE,
                       shuffle=True,  num_workers=2, pin_memory=True)
    dl_vl = DataLoader(ds_vl, batch_size=BATCH_SIZE,
                       shuffle=False, num_workers=2, pin_memory=True)
    return dl_tr, dl_vl

Create the 2-layer MLP

In [34]:
class CubagemMLP(nn.Module):
    """
    Two-layer MLP predicting 4 targets (3 dimensions + weight, in log space)
    from concatenated image/text embeddings + numerical/categorical/regex feats.
    """
    def __init__(
        self,
        emb_texto_dim,
        emb_img_dim,
        num_feats_dim,
        cat_feats_dim,
        regex_feats_dim,
        hidden_dim=2048,
        n_targets=4,
        dropout=0.5,
    ):
        super().__init__()
 
        input_dim = (
            emb_texto_dim + emb_img_dim + num_feats_dim + cat_feats_dim + regex_feats_dim
        )
 
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim//4, n_targets),
        )
 
    def forward(self, batch):
        x = torch.cat(
            [
                batch["emb_texto"],
                batch["emb_img"],
                batch["num_feats"],
                batch["cat_feats"],
                batch["regex_feats"],
            ],
            dim=1,
        )
        return self.net(x)

Training helper functions

In [42]:
import torch
import torch.nn as nn

@torch.no_grad()
def compute_rmse(preds, targets):
    """
    RMSE metric for monitoring
    """
    return torch.sqrt(torch.mean((preds - targets) ** 2)).item()

def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
 
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        targets = batch["targets"]
 
        optimizer.zero_grad()
        preds = model(batch)
        loss = loss_fn(preds, targets)  # MSE
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item()
        n_batches += 1
 
    return total_loss / n_batches

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_rmse = 0.0
    n_batches = 0
 
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        targets = batch["targets"]
 
        preds = model(batch)
        loss = loss_fn(preds, targets)
 
        rmse = compute_rmse(preds, targets)
 
        total_loss += loss.item()
        total_rmse += rmse
        n_batches += 1
 
    return {
        "loss": total_loss / n_batches,
        "rmse": total_rmse / n_batches,
    }

def train(
    model,
    train_loader,
    val_loader,
    n_epochs=50,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    device=None,
):
    """
    Training loop.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
  
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss() 
 
    history = []
    best_val_rmse = float("inf")
    best_state_dict = None
 
    for epoch in range(1, n_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate(model, val_loader, loss_fn, device)
 
        history.append(
            {"epoch": epoch, "train_mse": train_loss, **val_metrics}
        )
 
        # Track best RMSE and save weights
        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
 
        # Print logs
        if epoch % 5 == 0 or epoch == 1:
            msg = (
                f"Epoch {epoch:3d} | "
                f"train MSE: {train_loss:.4f} | "
                f"val MSE: {val_metrics['loss']:.4f} | "
                f"val RMSE: {val_metrics['rmse']:.4f}"
            )
            print(msg)
 
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
 
    return model, history

In [36]:
print("Missing numericals:", df[features["numerical"]].isna().sum().sum())
print("Missing categoricals:", df_encoded[OHE_columns].isna().sum().sum())

Missing numericals: 19526
Missing categoricals: 0


In [41]:
dl_train, dl_val = create_loaders(df_train, df_val)

dims = dict(
    emb_texto_dim=768,
    emb_img_dim=768,
    num_feats_dim=len(features["numerical"]),
    cat_feats_dim=len(OHE_columns),
    regex_feats_dim=len(features["title_regex"]),
)

model = CubagemMLP(**dims)

trained_model, history = train(
    model, dl_train, dl_val, n_epochs=100, batch_size=32, lr=1e-3, log_space=False
)
print("\nFinal history entry:", history[-1])

Epoch   1 | train MSE: 10.2600 | val MSE: 1.2675 | val RMSE (log-space): 1.0987
Epoch   5 | train MSE: 1.1112 | val MSE: 0.6743 | val RMSE (log-space): 0.8162
Epoch  10 | train MSE: 0.6580 | val MSE: 0.5853 | val RMSE (log-space): 0.7609
Epoch  15 | train MSE: 1.1835 | val MSE: 0.6041 | val RMSE (log-space): 0.7728
Epoch  20 | train MSE: 2.2347 | val MSE: 0.6950 | val RMSE (log-space): 0.8293
Epoch  25 | train MSE: 1.3860 | val MSE: 0.6700 | val RMSE (log-space): 0.8142
Epoch  30 | train MSE: 0.6169 | val MSE: 0.6989 | val RMSE (log-space): 0.8317
Epoch  35 | train MSE: 0.6830 | val MSE: 0.6987 | val RMSE (log-space): 0.8312
Epoch  40 | train MSE: 0.7244 | val MSE: 0.6931 | val RMSE (log-space): 0.8279
Epoch  45 | train MSE: 0.6486 | val MSE: 0.6792 | val RMSE (log-space): 0.8196
Epoch  50 | train MSE: 0.6786 | val MSE: 0.7379 | val RMSE (log-space): 0.8544
Epoch  55 | train MSE: 0.8843 | val MSE: 0.7305 | val RMSE (log-space): 0.8502
Epoch  60 | train MSE: 0.6241 | val MSE: 0.7554 | v

In [45]:
# Saving the model weights
torch.save(trained_model.state_dict(), "../weights/CubagemMLP.pth")

In [47]:
# Load the model
saved_weights = torch.load("../weights/CubagemMLP.pth")

model = CubagemMLP(**dims)
model.load_state_dict(saved_weights)

<All keys matched successfully>

O que ainda precisa fazer? Checar esse valor de MSE e plotar os graficos, alem de conferir na mao se tudo esta certo